# 一个异常读数，会把平均值拉偏多少？

**实验手记 01**　·　约 5 秒的小比较　·　本地 CPU

> 一支温度计连续读数九次，最后一次突然跳高。你会相信九次的平均数，还是排序后居中的那一次？

**阅读路线**　先看真实读数 → 对照误差图 → 改一个参数再试。

在 Python 3 内核中选择 **Restart Kernel and Run All Cells**。绘图使用 Matplotlib，逐格运行可按 **Shift+Enter**。

## ① 调一次偏差

把 `offset` 从 **8 °C** 改成 **0 °C**，就能用同一批读数做第二次比较。

In [ ]:
from random import Random
from statistics import mean, median
from time import perf_counter
from IPython.display import Markdown, display

reference = 20.0
n_readings = 9
offset = 8.0  # 改成 0，可看同一组读数没有人为异常时的结果
seed = 7

## ② 找到变化的读数

只看最后一次的前后值。其余八个读数在两组中相同。

In [ ]:
rng = Random(seed)
clean = [round(reference + rng.uniform(-0.4, 0.4), 2) for _ in range(n_readings)]
changed = clean.copy()
changed[-1] += offset
print("原始读数（°C）：", clean)
print(f"最后一次：{clean[-1]:.2f} → {changed[-1]:.2f} °C")

## ③ 哪种估计更接近 20 °C？

每根条表示估计温度与参考温度之间的距离。条越短，误差越小；表格给出相应数值。

In [ ]:
started = perf_counter()
labels = ("原始", "指定偏差")
rows = []
for condition, values in zip(labels, (clean, changed)):
    for name, estimate in (("平均值", mean(values)), ("中位数", median(values))):
        rows.append((condition, name, estimate, abs(estimate - reference)))

table = ["| 条件 | 方法 | 估计值（°C） | 绝对误差（°C）↓ |", "|:---|:---|---:|---:|"]
for condition, name, estimate, error in rows:
    table.append(f"| {condition} | {name} | {estimate:.3f} | {error:.3f} |")
elapsed_s = perf_counter() - started
from io import BytesIO
from matplotlib.figure import Figure
from IPython.display import Image

figure = Figure(figsize=(7, 2.8), layout="constrained")
axis = figure.subplots()
labels = ["Original / Mean", "Original / Median", "Changed / Mean", "Changed / Median"]
errors = [row[3] for row in rows]
bars = axis.barh(labels, errors, height=0.52, color=["#718598", "#248075"] * 2)
axis.invert_yaxis()
axis.bar_label(bars, fmt="%.3f", padding=5)
axis.set_xlim(0, max(errors + [0.01]) * 1.25)
axis.set_xlabel("Absolute error (°C)")
axis.spines[["top", "right", "left"]].set_visible(False)
axis.tick_params(axis="y", length=0)
buffer = BytesIO()
figure.savefig(buffer, format="png", dpi=130)
display(Image(data=buffer.getvalue()))
display(Markdown("\n".join(table)))

## 04　这个结果能说明多少？

下面的观察直接由表格计算。改参数会改变观察；“哪种方法在真实仪器上更好”仍需独立数据验证。

In [ ]:
error_mean, error_median = rows[2][3], rows[3][3]
comparison = ("中位数误差较小" if error_median < error_mean else
              "平均值误差较小" if error_mean < error_median else "两者误差相同")
print(f"实测观察：在指定偏差 {offset:g} °C 的这一组数据上，{comparison}。")
print(f"两种方法的绝对误差相差 {abs(error_mean - error_median):.3f} °C。")
print(f"本次比较与制表计算：{elapsed_s * 1000:.3f} ms（不含启动、输入准备和显示）。")

**解释与边界**　这组数字让我们看到单个偏高读数怎样影响估计。它是已知参考值下的合成示例，仅覆盖当前样本和参数。

**下一步**　如果关心这个观察是否只属于这一组样本，可以先保持条件、更换独立种子。若问题转为真实测量，应使用有可靠参考温度的独立数据。

**复核入口**　第 1 节记录生效参数，第 2 节生成全部输入，第 3 节的 `rows` 保留每一项未舍入的结果。